# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [2]:
# Clone your specific repository
!git clone https://github.com/PrathamDudani/FlyRank_Assignment.git

# Change into the repository's folder
%cd FlyRank_Assignment

Cloning into 'FlyRank_Assignment'...
remote: Enumerating objects: 132, done.
remote: Counting objects: 100% (132/132), done.
remote: Compressing objects: 100% (102/102), done.
remote: Total 132 (delta 39), reused 77 (delta 13), pack-reused 0 (from 0)
Receiving objects: 100% (132/132), 1.94 MiB | 8.43 MiB/s, done.
Resolving deltas: 100% (39/39), done.
/content/FlyRank_Assignment


In [3]:
import pandas as pd
import numpy as np
import os

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
valid = df[df["avg_position"] > 0].copy()   # avg_position=0 means no data — must filter

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [4]:
REASON_CODES = ["CTR_BELOW_POSITION_EXPECTED", "NO_FLAG"]

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [10]:
print(df.columns.tolist())

['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


In [11]:
# ── SECTION 2: Build the ranked queue ────────────────────────

# signal 1 bucket table (position vs ctr) — using your own bins, not the pre-built tier
valid["position_bucket"] = pd.cut(
    valid["avg_position"], bins=[0,3,6,10,20,1000],
    labels=["1-3","4-6","7-10","11-20","20+"]
)
signal1_table = valid.groupby("position_bucket")["ctr"].agg(["mean","median","count"])
print(signal1_table)

# signal 2 bucket table (staleness vs engagement)
valid["staleness_bucket"] = pd.cut(
    valid["days_since_last_update"], bins=[0,90,180,365,10000],
    labels=["<90d","90-180d","180-365d","365d+"]
)
signal2_table = valid.groupby("staleness_bucket")["engagement_rate"].agg(["mean","median","count"])
print(signal2_table)

# --- read the two tables above, THEN set these from what they actually show ---
expected_ctr = valid.groupby("position_bucket")["ctr"].transform("mean")
ctr_gap = expected_ctr - valid["ctr"]

GAP_THRESHOLD = None          # e.g. set to something like the median gap or a spread you saw
IMPRESSION_THRESHOLD = None   # e.g. set from valid["impressions_90d"].describe()

underperforming = (ctr_gap > GAP_THRESHOLD).astype(int)
visible = (valid["impressions_90d"] >= IMPRESSION_THRESHOLD).astype(int)

valid["score"] = underperforming * visible * valid["impressions_90d"]
valid["reason_code"] = np.where(
    (underperforming == 1) & (visible == 1),
    "CTR_BELOW_POSITION_EXPECTED", "NO_FLAG"
)
valid["action"] = np.where(valid["score"] > 0, "REVIEW_FOR_REFRESH", "NO_ACTION")

ranked = valid.sort_values("score", ascending=False)
os.makedirs("work/outputs", exist_ok=True)
ranked.to_csv("work/outputs/baseline_action_score.csv", index=False)

/tmp/ipykernel_2655/4242733187.py:8: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  signal1_table = valid.groupby("position_bucket")["ctr"].agg(["mean","median","count"])
/tmp/ipykernel_2655/4242733187.py:16: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  signal2_table = valid.groupby("staleness_bucket")["engagement_rate"].agg(["mean","median","count"])
/tmp/ipykernel_2655/4242733187.py:20: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence th

                     mean  median  count
position_bucket                         
1-3              2.714303    0.00   1141
4-6              0.931510    0.21   4801
7-10             0.459807    0.12   7041
11-20            0.323443    0.10   7273
20+              0.211333    0.00   8539
                      mean  median  count
staleness_bucket                         
<90d              2.686162     0.0  19475
90-180d           2.408497     0.0   9162
180-365d          2.306732     0.0    153
365d+             0.000000     0.0      5


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [12]:
top20 = ranked.head(20)
print(top20[["content_id","action","reason_code","score"]])

                 content_id     action reason_code  score
29999  content_887020f20b5e  NO_ACTION     NO_FLAG      0
0      content_304f48230142  NO_ACTION     NO_FLAG      0
1      content_a1fb4e703a9e  NO_ACTION     NO_FLAG      0
2      content_9aa793d4d895  NO_ACTION     NO_FLAG      0
3      content_331d6c4de07b  NO_ACTION     NO_FLAG      0
4      content_d99b7a2d90ca  NO_ACTION     NO_FLAG      0
5      content_d4084a4bc775  NO_ACTION     NO_FLAG      0
6      content_9a34b442b552  NO_ACTION     NO_FLAG      0
7      content_a63219c6e95a  NO_ACTION     NO_FLAG      0
8      content_5e6c160719bc  NO_ACTION     NO_FLAG      0
9      content_c27558df2b0c  NO_ACTION     NO_FLAG      0
29981  content_2dfd17269502  NO_ACTION     NO_FLAG      0
29980  content_81a91fe32bc2  NO_ACTION     NO_FLAG      0
29979  content_6f2f3043b633  NO_ACTION     NO_FLAG      0
29978  content_3dc420aa9809  NO_ACTION     NO_FLAG      0
29977  content_c87291853cab  NO_ACTION     NO_FLAG      0
29976  content

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [13]:
leak_cols = ["trend_direction", "trend_pct", "is_declining_label"]
used_cols = ["ctr", "avg_position", "impressions", "engagement_rate", "days_since_update"]
assert not set(leak_cols) & set(used_cols)
print("No leakage columns used.")

No leakage columns used.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.